In [1]:
# Import module constrained_likelihood_surrogates
#
import sys
sys.path.append("../src/")
from constrained_likelihood_surrogates import *

In [ ]:
# Mean and variance

In [2]:
# Investigating mean and variance of some estimates of statistics

# Checking distribution of statistics under constrained and typical surrogates:
# 
# Considers several types of surrogates:
# Constrained, typical, true, original, bootstrapped
# 
test_type = 'statistics'
lower_cutoff = 1
for N in [2**n for n in range(2, 11)]:
    # num_trans = min(2**16, N**2)
    # num_trans = max(num_trans, 2**6*N)
    num_trans = N*(int(np.ceil(np.log2(N*1024))))
    
    num_surr = 10**1
    num_tests = 10**1
    
    a = lower_cutoff
    b = 9
    
    stat_list = [mean_val, second_mom, third_mom, fourth_mom, max_val, variance, coef_var, dispersion, skew, kurtosis, evi_mom, evi_smooth]
    stat_name_list = [stat.__name__ for stat in stat_list]
    num_stats = len(stat_list)
    
    print('N = ' + str(N) + ', lower cut-off = ' + str(lower_cutoff))
    
    print(str(num_tests) + ' tests, each with ' + str(num_surr) + ' constrained, typical surrogates, each using ' + str(num_trans) + ' transitions')
    
    # process_list = ['powerlaw', 'lognorm', 'expon', 'truncnorm', 'uniform', 'powerlaw_t', 'lognorm_t', 'expon_t', 'truncnorm_t', 'uniform_t']
    process_list = ['powerlaw', 'lognorm', 'expon', 'truncnorm', 'uniform']
    # process_list = ['powerlaw', 'lognorm', 'uniform']
    # process_list = ['expon', 'truncnorm', 'uniform']
    num_processes = len(process_list)
    
    method_list = ['constrained', 'typical', 'original', 'true', 'bootstrap']
    num_methods = len(method_list)#Based on constrained, typical, true, original, bootstrap
    
    mean_stat_list_list_list = np.full((num_methods, num_processes, num_tests, num_stats), np.nan)#Array of nans: num_methods x num_processes x num_statistics x num_tests x num_statistics
    var_stat_list_list_list = np.full((num_methods, num_processes, num_tests, num_stats), np.nan)#Array of nans: num_methods x num_processes x num_statistics x num_tests x num_statistics

    save_str_0 = test_type + '_xmin-' + str(lower_cutoff) + '_N-' + str(N) + '_ntra-' + str(num_trans) + '_nsur-' + str(num_surr) + '_ntes-' + str(num_tests) + '_ntyp-' + str(num_methods) + '_npro-' + str(num_processes) + '_nstats-' + str(num_stats)

    process_str = ''
    process_dict = {}
    process_time_list = []

    lower_cutoff_hat = lower_cutoff

    for i_process in range(num_processes):
        start_time_process = timer()
        process = process_list[i_process]

        if (process == 'powerlaw'):#Generate power law sequence
            alpha = 1.5
            param = [alpha]
            param_str = '[alpha]'

        elif (process == 'lognorm'):#Generate truncated log normal sequence
            mu = -1
            sigma = 1
            param = [mu, sigma]
            param_str = '[mu/alpha, sigma]'

        elif (process == 'expon'):#Generate exponential sequence
            lam = 1
            param = [lam]
            param_str = '[lambda]'

        elif (process == 'truncnorm'):#Generate truncated normal sequence
            mu = -1
            sigma = 1
            param = [mu, sigma]
            param_str = '[mu/lambda, sigma]'

        elif (process == 'uniform'):#Generate uniform sequence
            param = [b]
            param_str = '[b]'

        if (process == 'powerlaw_t'):#Generate tightly bounded power law sequence
            alpha = 1.5
            param = [alpha, a, b]
            param_str = '[alpha, a, b]'

        elif (process == 'lognorm_t'):#Generate tightly bounded truncated log normal sequence
            mu = -1
            sigma = 1
            param = [mu, sigma, a, b]
            param_str = '[mu/alpha, sigma, a, b]'

        elif (process == 'expon_t'):#Generate tightly bounded exponential sequence
            lam = 1
            param = [lam, a, b]
            param_str = '[lambda, a, b]'

        elif (process == 'truncnorm_t'):#Generate tightly bounded truncated normal sequence
            mu = -1
            sigma = 1
            param = [mu, sigma, a, b]
            param_str = '[mu/lambda, sigma, a, b]'

        elif (process == 'uniform_t'):#Generate tightly bounded uniform sequence
            param = [b, a, b]
            param_str = '[b, a, b]'

        print('Process is ' + process)
        print('True ' + param_str + ' = ' + str(param))

        process_dict[process] = [param, param_str]

        num_params = len(param)

        param_str_2 = str(param)
        param_str_2 = param_str_2.replace(', ', '-')
        param_str_2 = param_str_2[1:-1]
        if (i_process < num_processes):
            process_str = process_str + '_'
        process_str = process_str + process[0] + '-' + param_str_2

        mean_stat_list_list = np.full((num_methods, num_tests, num_stats), np.nan)#Array of nans
        var_stat_list_list = np.full((num_methods, num_tests, num_stats), np.nan)#Array of nans

        for i_test in range(num_tests):
            #Generate time series and calculate ks distance:
            val_seq = gen_data(process, N=N, lower_cutoff=lower_cutoff, param=param)
            param_hat_p, lp_seq_p = fit_model(val_seq, lower_cutoff_hat=lower_cutoff_hat, model=process)#p for (true) process
            ks_dist_p = calc_ks(val_seq, model=process, lower_cutoff_hat=lower_cutoff_hat, param_hat=param_hat_p)#m for model

            c_surr_p_stat_val_list_list = np.zeros(shape=(num_surr, num_stats))#Constrained
            t_surr_p_stat_val_list_list = np.zeros(shape=(num_surr, num_stats))#Typical
            tr_surr_p_stat_val_list_list = np.zeros(shape=(num_surr, num_stats))#True
            o_surr_p_stat_val_list_list = np.zeros(shape=(num_surr, num_stats))#Original
            b_surr_p_stat_val_list_list = np.zeros(shape=(num_surr, num_stats))#Bootstrapped

            surr_p_val_seq = val_seq
            surr_p_val_seq_list = []
            for i_surr in range(num_surr):
                method = 'constrained'
                surr_p_val_seq = gen_surrogate(surr_p_val_seq, model=process, method=method, num_trans=num_trans, lower_cutoff_hat=lower_cutoff_hat, param_hat=param_hat_p)
                random.shuffle(surr_p_val_seq)
                surr_p_val_seq_list = surr_p_val_seq_list + [surr_p_val_seq]

            for i_surr in range(num_surr):
                method = 'constrained'
                surr_p_val_seq = surr_p_val_seq_list[i_surr]
                surr_p_param_hat_p, surr_p_lp_seq_p = fit_model(surr_p_val_seq, lower_cutoff_hat=lower_cutoff_hat, model=process)#p for (true) process
                surr_p_ks_dist_p = calc_ks(surr_p_val_seq, model=process, lower_cutoff_hat=lower_cutoff_hat, param_hat=surr_p_param_hat_p)#Corrected 'param_hat_p' to 'surr_p_param_hat_p' on 2025.06.27#Corrected 'param_hat_p' to 'surr_p_param_hat_p' on 2025.06.27
                surr_p_stat_val_list = [stat(surr_p_val_seq) for stat in stat_list]
                c_surr_p_stat_val_list_list[i_surr, :] = surr_p_stat_val_list

            for i_surr in range(num_surr):
                method = 'typical'
                surr_p_val_seq = gen_surrogate(val_seq, model=process, method=method, num_trans=num_trans, lower_cutoff_hat=lower_cutoff_hat, param_hat=param_hat_p)
                surr_p_param_hat_p, surr_p_lp_seq_p = fit_model(surr_p_val_seq, lower_cutoff_hat=lower_cutoff_hat, model=process)#p for (true) process
                surr_p_ks_dist_p = calc_ks(surr_p_val_seq, model=process, lower_cutoff_hat=lower_cutoff_hat, param_hat=surr_p_param_hat_p)#Corrected 'param_hat_p' to 'surr_p_param_hat_p' on 2025.06.27#Corrected 'param_hat_p' to 'surr_p_param_hat_p' on 2025.06.27
                surr_p_stat_val_list = [stat(surr_p_val_seq) for stat in stat_list]
                t_surr_p_stat_val_list_list[i_surr, :] = surr_p_stat_val_list

            for i_surr in range(num_surr):
                method = 'original'
                surr_p_val_seq = val_seq
                surr_p_param_hat_p, surr_p_lp_seq_p = param_hat_p, lp_seq_p
                surr_p_ks_dist_p = ks_dist_p
                surr_p_stat_val_list = [stat(surr_p_val_seq) for stat in stat_list]
                o_surr_p_stat_val_list_list[i_surr, :] = surr_p_stat_val_list

            for i_surr in range(num_surr):
                method = 'true'
                surr_p_val_seq = gen_data(process, N=N, lower_cutoff=lower_cutoff, param=param)
                surr_p_param_hat_p, surr_p_lp_seq_p = fit_model(surr_p_val_seq, lower_cutoff_hat=lower_cutoff, model=process)#p for (true) process
                surr_p_ks_dist_p = calc_ks(surr_p_val_seq, model=process, lower_cutoff_hat=lower_cutoff, param_hat=surr_p_param_hat_p)#Corrected 'param_hat_p' to 'surr_p_param_hat_p' on 2025.06.27#Corrected 'param_hat_p' to 'surr_p_param_hat_p' on 2025.06.27
                surr_p_stat_val_list = [stat(surr_p_val_seq) for stat in stat_list]
                tr_surr_p_stat_val_list_list[i_surr, :] = surr_p_stat_val_list

            for i_surr in range(num_surr):
                method = 'bootstrap'
                surr_p_val_seq = bootstrap(val_seq, N=N)
                surr_p_param_hat_p, surr_p_lp_seq_p = fit_model(surr_p_val_seq, lower_cutoff_hat=lower_cutoff_hat, model=process)#p for (true) process
                surr_p_param_hat_p, surr_p_lp_seq_p = fit_model(surr_p_val_seq, lower_cutoff_hat=lower_cutoff_hat, model=process)
                surr_p_ks_dist_p = calc_ks(surr_p_val_seq, model=process, lower_cutoff_hat=lower_cutoff_hat, param_hat=surr_p_param_hat_p)#Corrected 'param_hat_p' to 'surr_p_param_hat_p' on 2025.06.27#Corrected 'param_hat_p' to 'surr_p_param_hat_p' on 2025.06.27
                surr_p_stat_val_list = [stat(surr_p_val_seq) for stat in stat_list]
                b_surr_p_stat_val_list_list[i_surr, :] = surr_p_stat_val_list

            for i_method in range(num_methods):
                method = method_list[i_method]
                if (method == 'constrained'):#Constrained surrogates
                    stat_val_surr_p_list_list = c_surr_p_stat_val_list_list
                elif (method == 'typical'):#Typical surrogates
                    stat_val_surr_p_list_list = t_surr_p_stat_val_list_list
                elif (method == 'original'):#Original sequence
                    stat_val_surr_p_list_list = o_surr_p_stat_val_list_list
                elif (method == 'true'):#Surrogates based on true process
                    stat_val_surr_p_list_list = tr_surr_p_stat_val_list_list
                elif (method == 'bootstrap'):#Bootstrapping
                    stat_val_surr_p_list_list = b_surr_p_stat_val_list_list
                else:
                    raise Exception('I am not prepared for this surrogate method.')
                mean_stat_list_list[i_method, i_test, :] = np.mean(stat_val_surr_p_list_list, axis=0)
                var_stat_list_list[i_method, i_test, :] = np.var(stat_val_surr_p_list_list, axis=0)
                mean_stat_list_list_list[i_method, i_process, i_test, :] = np.mean(stat_val_surr_p_list_list, axis=0)
                var_stat_list_list_list[i_method, i_process, i_test, :] = np.var(stat_val_surr_p_list_list, axis=0)

        end_time_process = timer()
        total_time_process = end_time_process - start_time_process
        process_time_list = process_time_list + [total_time_process]
        print('That took ' + str(total_time_process) + 'sec.') # Time in seconds, e.g. 5.38091952400282

    save_str = save_str_0 + process_str

    save_dict = {'mean_stat_list_list_list':mean_stat_list_list_list.tolist(),
                 'var_stat_list_list_list':var_stat_list_list_list.tolist(),
                 'test_type':test_type,
                 'lower_cutoff':lower_cutoff,
                 'N':N,
                 'num_trans':num_trans,
                 'num_surr':num_surr,
                 'num_tests':num_tests,
                 'method_list':method_list,
                 'num_methods':num_methods,
                 'process_list':process_list,
                 'process_dict':process_dict,
                 'process_str':process_str,
                 'stat_name_list':stat_name_list,
                 'save_str':save_str,
                 'process_time_list':process_time_list,  
    }

    save('./results/est-stat/' + save_str, save_dict)

N = 4, lower cut-off = 1
10 tests, each with 10 constrained, typical surrogates, each using 48 transitions
Process is powerlaw
True [alpha] = [1.5]


C:\Users\murdo\AppData\Local\Temp\ipykernel_27500\2692168840.py:180: RuntimeWarning: Precision loss occurred in moment calculation due to catastrophic cancellation. This occurs when the data are nearly identical. Results may be unreliable.
  surr_p_stat_val_list = [stat(surr_p_val_seq) for stat in stat_list]


That took 0.9246910000001662sec.
Process is lognorm
True [mu/alpha, sigma] = [-1, 1]
That took 6.423548699998719sec.
Process is expon
True [lambda] = [1]
That took 0.8821982000008575sec.
Process is truncnorm
True [mu/lambda, sigma] = [-1, 1]
That took 7.022082400000727sec.
Process is uniform
True [b] = [9]
That took 0.5623440999988816sec.
N = 8, lower cut-off = 1
10 tests, each with 10 constrained, typical surrogates, each using 104 transitions
Process is powerlaw
True [alpha] = [1.5]
That took 1.0192621000005602sec.
Process is lognorm
True [mu/alpha, sigma] = [-1, 1]
That took 6.511158299999806sec.
Process is expon
True [lambda] = [1]
That took 0.9176065999999992sec.
Process is truncnorm
True [mu/lambda, sigma] = [-1, 1]
That took 6.1259322999994765sec.
Process is uniform
True [b] = [9]
That took 0.5762293999996473sec.
N = 16, lower cut-off = 1
10 tests, each with 10 constrained, typical surrogates, each using 224 transitions
Process is powerlaw
True [alpha] = [1.5]
That took 1.092068